## `insights.ipynb - Step 3 of  the pipeline`
**Role**:   Adds cluster labels (per product, from product embeddings) and sentiment labels (per review,from star rating) to the dataset.                           
**Input**:  `reviews_df.pkl`, `products_df.pkl`.                                            
**Output**: `reviews_insights.pkl`, `products_insights.pkl`.                     

`Be aware to change the "DIRECTORY" variable (cell 3) with the right path of the folder on Google Drive where you uploaded the dataset!`

The `.pkl` format is sensitive to the Pandas version: the version used to create the file must match the one used to read it. Since Google Colab may ship a different Pandas version than the one on your local machine, run the cell below to install the exact version used during preprocessing.

In [1]:
!pip install pandas==3.0.1

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Set DIRECTORY to the Google Drive folder where you uploaded `reviews_df.pkl` and `products_df`.pkl*                  
*Examples*:
```python
# Dataset is directly in MyDrive root:
DIRECTORY = HERE / "drive" / "MyDrive"

# Dataset is inside a subfolder called "Data Science in Action":
DIRECTORY = HERE / "drive" / "MyDrive" / "Data Science in Action"
```

In [3]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.cluster import MiniBatchKMeans
from transformers import pipeline

# --------------------------------------------------
# Paths
# --------------------------------------------------
HERE = Path.cwd()
DIRECTORY = HERE / "drive" / "MyDrive" / "Data Science in Action"  

REVIEWS_INPUT_PATH = DIRECTORY / "reviews_df.pkl"
PRODUCTS_INPUT_PATH = DIRECTORY / "products_df.pkl"

REVIEWS_OUTPUT_PATH = DIRECTORY / "reviews_insights.pkl"
PRODUCTS_OUTPUT_PATH = DIRECTORY / "products_insights.pkl"

In [4]:
# --------------------------------------------------
# Config
# --------------------------------------------------
N_CLUSTERS = 200
SENTIMENT_MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment-latest"
BATCH_SIZE = 256

In [5]:
# --------------------------------------------------
# Load data
# --------------------------------------------------
def load_data(reviews_path: Path, products_path: Path):
    print("Loading reviews...")
    try:
        reviews_df = pd.read_pickle(reviews_path)
        print(f"  {len(reviews_df):,} reviews loaded")
    except FileNotFoundError as e:
        raise FileNotFoundError(f"Failed to load reviews. Check if the path {reviews_path} is correct.") from e
    except (ModuleNotFoundError, ImportError) as e:
            raise ImportError(f"Version mismatch loading reviews.\n"
                "Did you run the '!pip install pandas==3.0.1' cell?") from e
    
    print("Loading products...")
    try:
        products_df = pd.read_pickle(products_path)
        print(f"  {len(products_df):,} products loaded")
    except FileNotFoundError as e:
        raise FileNotFoundError(f"Failed to load products. Check if the path {products_path} is correct.") from e
    except (ModuleNotFoundError, ImportError) as e:
            raise ImportError(f"Version mismatch loading products.\n"
                "Did you run the '!pip install pandas==3.0.1' cell?") from e

    return reviews_df, products_df

In [6]:
# --------------------------------------------------
# Cluster products based on product embeddings
# --------------------------------------------------
def cluster_products(products_df: pd.DataFrame, n_clusters: int) -> pd.DataFrame:
    print(f"Clustering {len(products_df):,} products into {n_clusters} clusters...")
    embeddings = np.stack(products_df["product_embedding"].values)
    model = MiniBatchKMeans(n_clusters=n_clusters, random_state=42, n_init=3)
    products_df = products_df.copy()
    products_df["cluster_label"] = model.fit_predict(embeddings)
    print("Clustering done.")
    return products_df

In [7]:
def add_sentiment(reviews_df: pd.DataFrame, batch_size: int = 256) -> pd.DataFrame:
    """Run a 3-class sentiment model on review text."""
    reviews_df = reviews_df.copy()

    classifier = pipeline("text-classification", model=SENTIMENT_MODEL_NAME,
                          device=0, truncation=True, max_length=512)
    texts = reviews_df["text"].fillna("").tolist()
    print(f"Running sentiment on {len(texts):,} reviews (batch={batch_size})...")

    results = classifier(texts, batch_size=batch_size)
    # model outputs: "positive", "negative", "neutral"
    reviews_df["sentiment_label"] = [r["label"].lower() for r in results]

    return reviews_df

In [8]:
# --------------------------------------------------
# Propagate cluster labels from products to reviews
# --------------------------------------------------
def add_cluster_to_reviews(reviews_df: pd.DataFrame, products_df: pd.DataFrame) -> pd.DataFrame:
    print("Propagating cluster labels to reviews...")
    cluster_map = products_df.set_index("parent_asin")["cluster_label"].to_dict()
    reviews_df = reviews_df.copy()
    reviews_df["cluster_label"] = reviews_df["parent_asin"].map(cluster_map)
    return reviews_df

In [9]:
# --------------------------------------------------
# Pipeline
# --------------------------------------------------
def run_insights(
    reviews_input_path: Path = REVIEWS_INPUT_PATH,
    products_input_path: Path = PRODUCTS_INPUT_PATH,
    reviews_output_path: Path = REVIEWS_OUTPUT_PATH,
    products_output_path: Path = PRODUCTS_OUTPUT_PATH,
    n_clusters: int  = N_CLUSTERS,
    batch_size:  int = BATCH_SIZE
):
    reviews_df, products_df = load_data(reviews_input_path, products_input_path)

    products_df = cluster_products(products_df, n_clusters)
    reviews_df = add_sentiment(reviews_df, batch_size=batch_size)
    reviews_df = add_cluster_to_reviews(reviews_df, products_df)

    reviews_output_path.parent.mkdir(parents=True, exist_ok=True)
    reviews_df.to_pickle(reviews_output_path)
    print(f"Reviews saved: {reviews_df.shape} → {reviews_output_path}")

    products_df.to_pickle(products_output_path)
    print(f"Products saved: {products_df.shape} → {products_output_path}")

    return reviews_df, products_df

In [10]:
run_insights()

Loading reviews...
  488,964 reviews loaded
Loading products...
  60,272 products loaded
Clustering 60,272 products into 50 clusters...
Clustering done.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


... (2 outputs left out)

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Running sentiment on 488,964 reviews (batch=256)...
Propagating cluster labels to reviews...
Reviews saved: (488964, 11) → /content/drive/MyDrive/Data Science in Action/reviews_insights.pkl
Products saved: (60272, 12) → /content/drive/MyDrive/Data Science in Action/products_insights.pkl


(        rating                                            title_x  \
 0            4  12 mg is 12 on the periodic table people! Mg f...   
 1            5                 Save the lanet using less plastic.   
 2            5                                          Fantastic   
 3            4  It holds the water and makes bubbles.  That's ...   
 4            1                                         Not for me   
 ...        ...                                                ...   
 494116       5                                        Best brush!   
 494117       5                           It makes my skin softer.   
 494118       5                            This brush is perfect !   
 494119       5                                        Easy to use   
 494120       5       La comunicación con la empresa y ez perfecto   
 
                                                      text  \
 0       This review is more to clarify someone else’s ...   
 1       Love these easy multitask